# SSE 前置课：从字节到事件，亲手拆一遍

> 这是 [`01_streaming_tool_call.md`](01_streaming_tool_call.md) 的**前置实验课**。
> 01 讲"整条链路谁负责什么"，这一课只做一件事：**把 SSE 拆开，再拼回来。**

**只用标准库，仅需提供API Key。** 所有现象都在你本机复现。

学完你应该能毫不犹豫地答出：

1. 一个 SSE 帧在字节上长什么样？边界是什么？
2. TCP 把字节切碎了，为什么不能把"一次 read"当成一条消息？
3. 中文为什么会在流式里炸 `UnicodeDecodeError`？
4. `input_json_delta` 为什么**不能**逐段 `json.loads`？
5. 流式事件和最终 message，哪个才是"真相"？
6. 这一整套东西，对应 `cli_agent` 里哪几行代码？
7. 换成 OpenAI 兼容协议（DeepSeek），你写的解析器要改几行？

**运行方式**：从上到下逐个运行（Shift+Enter）。
遇到 `assert` 不要跳过 —— 它们是本课的验收点。

第 1–8 节完全离线。**第 9 节会真的调用 DeepSeek**，需要 `DEEPSEEK_API_KEY`
（从环境变量或 `~/.bashrc` 读取）；找不到就自动跳过，不影响其余内容。


## 0 · 先把分界线画出来

上一课有一句核心结论：

> **远端负责"把模型输出组织成 API 语义结构，并编码成 SSE 帧"；**
> **本地 SDK 负责"把收到的 SSE 帧解析成本地事件对象"。**

这一课要你**自己扮演"本地 SDK"**。

| 谁做 | 做什么 | 本课怎么模拟 |
|---|---|---|
| 远端服务层 | token → 文本 → content block → `data: {...}\n\n` | 第 1 节，一个 3 行的 `sse()` 函数 |
| 网络 | 把字节**随便**切碎 | 第 3 节，随机切分 1000 次 |
| **你** | 字节 → 帧 → JSON → 事件 → 最终 message | 本课要写的全部代码 |

写完之后你会发现：这条"解析链"总共不到 80 行。
`cli_agent` 的 `anthropic_compat.py` 里也没有魔法 —— 它只是把这段代码交给了 SDK。


In [6]:
import codecs
import http.client
import json
import os
import pathlib
import random
import re
import socket
import ssl
import sys
import threading
import time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

print("Python:", sys.version.split()[0])

try:
    import cli_agent

    HAS_CLI_AGENT = True
    print("cli_agent:", cli_agent.__file__)
except ImportError as exc:  # 没装也能跑完第 7 节
    HAS_CLI_AGENT = False
    print("cli_agent 未安装，第 8/9 节会跳过：", exc)

Python: 3.14.4
cli_agent: /home/wpx/work/learn/agent/cli_agent/src/cli_agent/__init__.py


## 1 · SSE 的语法，只有四条规则

一个事件帧 = 若干 `field: value` 行 + **一个空行**结束。

| 字段 | 含义 | 本课的 Anthropic 流里 |
|---|---|---|
| `event:` | 事件类型（可选） | `content_block_delta` 等 |
| `data:` | 载荷，可多行，用 `\n` 拼成一份 | 一个 JSON 对象 |
| `id:` | 断线重连游标（可选） | 本课不用 |
| `retry:` | 重连间隔毫秒（可选） | 本课不用 |

真正要记的只有四条：

1. **空行是帧边界**：`\n\n`，也允许 `\r\n\r\n`。
2. `field: value` 冒号后**第一个空格**被吃掉，之后的空格保留。
3. 以 `:` 开头的行是**注释**（常被用作心跳 `: ping`），整行丢弃。
4. 一个帧里可以有多行 `data:`，它们用 `\n` 连接成一份载荷。

第 1 条是生死线，其余是细节。下面先把"远端"那一侧写出来。


In [7]:
# ---- 模拟"远端服务层"：把结构化数据编码成 SSE 帧 ----
def sse(event: str, data: dict) -> str:
    """远端做的事：加 event:/data: 前缀，末尾补一个空行。就这么简单。"""
    return f"event: {event}\ndata: {json.dumps(data, ensure_ascii=False)}\n\n"


COMMAND = "printf 'print(\"你好，world\")' > hello.py"
TOOL_JSON = json.dumps({"command": COMMAND}, ensure_ascii=False)

# 故意把工具参数切成 4 段：第 2 段是"孤立的反斜杠"（切在 \" 中间）
BS = TOOL_JSON.index("\\")      # 第一个反斜杠的位置
HAN = TOOL_JSON.index("你")     # 第一个汉字的位置
cuts = [0, BS, BS + 1, HAN + 1, len(TOOL_JSON)]
FRAGMENTS = [TOOL_JSON[a:b] for a, b in zip(cuts, cuts[1:])]

print("完整的工具参数 JSON：")
print(" ", TOOL_JSON)
print(f"\n被切成 {len(FRAGMENTS)} 段：")
for i, f in enumerate(FRAGMENTS):
    note = "   ← 一个孤立的反斜杠，单独的 \\ 不是合法 JSON" if f == "\\" else ""
    print(f"  [{i}] {f!r}{note}")
assert "".join(FRAGMENTS) == TOOL_JSON, "碎片拼起来必须等于原文"
print("\n✅ 拼回去和原文完全一致 —— 碎片没有丢信息，只是被切开了")

完整的工具参数 JSON：
  {"command": "printf 'print(\"你好，world\")' > hello.py"}

被切成 4 段：
  [0] '{"command": "printf \'print('
  [1] '\\'   ← 一个孤立的反斜杠，单独的 \ 不是合法 JSON
  [2] '"你'
  [3] '好，world\\")\' > hello.py"}'

✅ 拼回去和原文完全一致 —— 碎片没有丢信息，只是被切开了


### 1.1 一条完整的流

下面这条流，就是"创建 hello.py 并运行它"会产生的 **Anthropic 风格**响应。
字段名与真实 API 一致，内容是我们构造的。

注意结构：一个 `message` 里有**多个 content block**，每个 block 有**自己的 index**，
各自 start / delta / stop。文本块和工具块是**并列**的，不是二选一。

另外留意第 2 段碎片 `'\\'`：它是**一个孤立的反斜杠**。
在**字符串**层面切分时，一个汉字不会被切开（`'你'` 始终完整）；
"切开汉字"只发生在**字节**层面 —— 那是第 4 节的坑。


In [8]:
RAW = "".join([
    sse("message_start", {"type": "message_start", "message": {
        "id": "msg_01DEMO", "type": "message", "role": "assistant",
        "model": "demo-model", "content": [],
        "usage": {"input_tokens": 42, "output_tokens": 1},
    }}),

    # ---- 块 0：文本 ----
    sse("content_block_start", {"type": "content_block_start", "index": 0,
        "content_block": {"type": "text", "text": ""}}),
    sse("content_block_delta", {"type": "content_block_delta", "index": 0,
        "delta": {"type": "text_delta", "text": "我来创建 "}}),
    sse("content_block_delta", {"type": "content_block_delta", "index": 0,
        "delta": {"type": "text_delta", "text": "hello.py 并运行它。"}}),
    sse("content_block_stop", {"type": "content_block_stop", "index": 0}),

    # ---- 块 1：工具调用 ----
    sse("content_block_start", {"type": "content_block_start", "index": 1,
        "content_block": {"type": "tool_use", "id": "toolu_01DEMO",
                          "name": "Bash", "input": {}}}),
    *[sse("content_block_delta", {"type": "content_block_delta", "index": 1,
        "delta": {"type": "input_json_delta", "partial_json": frag}})
      for frag in FRAGMENTS],
    sse("content_block_stop", {"type": "content_block_stop", "index": 1}),

    # ---- 收尾 ----
    sse("message_delta", {"type": "message_delta",
        "delta": {"stop_reason": "tool_use", "stop_sequence": None},
        "usage": {"output_tokens": 37}}),
    sse("message_stop", {"type": "message_stop"}),
])

print(f"共 {len(RAW)} 个字符 / {len(RAW.encode('utf-8'))} 个字节   ← 注意两者不等，中文占 3 字节")
print("\n前 260 个字节（这就是网线上真正跑的东西）：")
print("-" * 70)
print(RAW.encode("utf-8")[:260])
print("-" * 70)

共 1735 个字符 / 1759 个字节   ← 注意两者不等，中文占 3 字节

前 260 个字节（这就是网线上真正跑的东西）：
----------------------------------------------------------------------
b'event: message_start\ndata: {"type": "message_start", "message": {"id": "msg_01DEMO", "type": "message", "role": "assistant", "model": "demo-model", "content": [], "usage": {"input_tokens": 42, "output_tokens": 1}}}\n\nevent: content_block_start\ndata: {"type": "c'
----------------------------------------------------------------------


## 2 · 写一个分帧器（10 行）

**这是全课唯一需要"聪明"的地方。**

一个分帧器只干一件事：不断往缓冲区追加文本，把缓冲区里所有**完整的帧**切走，剩下的留着等下一批。

关键心态：**你永远不知道下一次来的是什么**，所以你不能"解析"，只能"追加 + 尝试切分"。


In [31]:
FRAME_SEP = re.compile(r"\r?\n\r?\n")


class SSEFramer:
    """把"任意切分的文本"还原成"完整的 SSE 帧"。"""

    def __init__(self) -> None:
        self._buf = ""

    def feed(self, text: str) -> list[str]:
        self._buf += text
        out = []
        while (m := FRAME_SEP.search(self._buf)) is not None:
            out.append(self._buf[: m.start()])
            self._buf = self._buf[m.end():]
        return out

    @property
    def pending(self) -> str:
        """缓冲区里还没凑成完整帧的残料。正常情况下最后应为空。"""
        return self._buf


framer = SSEFramer()
frames = framer.feed(RAW)
print(f"切出 {len(frames)} 帧，缓冲区残留 {len(framer.pending)} 字符")
print()
for i, f in enumerate(frames):
    print(f"  帧{i:02d}: {f.splitlines()[0]}")

切出 13 帧，缓冲区残留 0 字符

  帧00: event: message_start
  帧01: event: content_block_start
  帧02: event: content_block_delta
  帧03: event: content_block_delta
  帧04: event: content_block_stop
  帧05: event: content_block_start
  帧06: event: content_block_delta
  帧07: event: content_block_delta
  帧08: event: content_block_delta
  帧09: event: content_block_delta
  帧10: event: content_block_stop
  帧11: event: message_delta
  帧12: event: message_stop


### 2.1 帧内解析

一个帧是若干 `field: value` 行。三件事要处理：注释行丢弃、多行 `data:` 用 `\n` 拼接、
冒号后只吃掉**一个**空格。


In [10]:
def parse_frame(frame: str) -> dict | None:
    """一个帧 → {"event": ..., "data": {...}}。纯注释帧返回 None。"""
    event_type: str | None = None
    data_lines: list[str] = []

    for line in frame.splitlines():
        if not line or line.startswith(":"):   # 空行 / 注释(心跳) → 丢
            continue
        field, _, value = line.partition(":")
        if value.startswith(" "):
            value = value[1:]                  # 只吃掉一个空格
        if field == "event":
            event_type = value
        elif field == "data":
            data_lines.append(value)
        # id: / retry: 本课忽略

    if not data_lines:
        return None                            # 纯注释帧，没有载荷
    return {"event": event_type, "data": json.loads("\n".join(data_lines))}


parsed = [parse_frame(f) for f in frames]
print(f"{'#':>2}  {'event:':<22} {'data.type':<22} 关键字段")
print("-" * 78)
for i, p in enumerate(parsed):
    ev = p["data"]
    if ev["type"] == "content_block_delta":
        d = ev["delta"]
        detail = d.get("text") or d.get("partial_json")
    elif ev["type"] == "content_block_start":
        detail = ev["content_block"]["type"]
    elif ev["type"] == "message_delta":
        detail = f"stop_reason={ev['delta']['stop_reason']}"
    else:
        detail = ""
    print(f"{i:>2}  {p['event']:<22} {ev['type']:<22} {detail!r}")

# SSE 允许纯注释帧（心跳），它没有 data:，parse_frame 应返回 None
print("\n心跳帧 parse_frame(': ping') →", parse_frame(": ping"), "  ← 整帧丢弃")

 #  event:                 data.type              关键字段
------------------------------------------------------------------------------
 0  message_start          message_start          ''
 1  content_block_start    content_block_start    'text'
 2  content_block_delta    content_block_delta    '我来创建 '
 3  content_block_delta    content_block_delta    'hello.py 并运行它。'
 4  content_block_stop     content_block_stop     ''
 5  content_block_start    content_block_start    'tool_use'
 6  content_block_delta    content_block_delta    '{"command": "printf \'print('
 7  content_block_delta    content_block_delta    '\\'
 8  content_block_delta    content_block_delta    '"你'
 9  content_block_delta    content_block_delta    '好，world\\")\' > hello.py"}'
10  content_block_stop     content_block_stop     ''
11  message_delta          message_delta          'stop_reason=tool_use'
12  message_stop           message_stop           ''

心跳帧 parse_frame(': ping') → None   ← 整帧丢弃


## 3 · 陷阱一：TCP 会把帧切碎

服务端写出去的是"帧"，你读到的却是"字节"。

TCP 只保证**顺序和完整性**，**不保证边界**。你一次 `recv()` 可能拿到半个帧，
也可能拿到三个半帧。

下面做一次暴力验证：把同一份字节，用九种方式喂给分帧器，看结果是否完全一致。

> **注意这里多加了一层 `ByteFramer`**：它的职责是"字节 → 文本"，用的是**增量解码器**。
> 为什么必须增量？第 4 节会用异常逼你看清楚。


In [11]:
class ByteFramer:
    """字节 → 帧。先增量解码 UTF-8，再交给文本分帧器。"""

    def __init__(self) -> None:
        self._decoder = codecs.getincrementaldecoder("utf-8")()
        self._framer = SSEFramer()

    def feed(self, chunk: bytes) -> list[str]:
        return self._framer.feed(self._decoder.decode(chunk))

    def close(self) -> list[str]:
        return self._framer.feed(self._decoder.decode(b"", final=True))


def frames_from_chunks(chunks: list[bytes]) -> list[str]:
    bf = ByteFramer()
    out: list[str] = []
    for c in chunks:
        out += bf.feed(c)
    return out + bf.close()


DATA = RAW.encode("utf-8")
baseline = frames_from_chunks([DATA])
assert baseline == frames, "整块喂进去，结果必须和上面一致"

cases = {
    "1 字节/次": [DATA[i:i + 1] for i in range(len(DATA))],
    "7 字节/次": [DATA[i:i + 7] for i in range(0, len(DATA), 7)],
    "97 字节/次": [DATA[i:i + 97] for i in range(0, len(DATA), 97)],
    "4096 字节/次": [DATA[i:i + 4096] for i in range(0, len(DATA), 4096)],
}
for label, chunks in cases.items():
    got = frames_from_chunks(chunks)
    ok = "一致 ✅" if got == baseline else "不一致 ❌"
    print(f"{label:<14}分成 {len(chunks):>5} 次喂入 → {len(got):>2} 帧 → {ok}")

rng = random.Random(0)
for trial in range(1000):
    n = rng.randrange(1, 40)
    cuts = sorted(rng.randrange(len(DATA) + 1) for _ in range(n))
    chunks, prev = [], 0
    for c in cuts:
        chunks.append(DATA[prev:c])
        prev = c
    chunks.append(DATA[prev:])
    assert frames_from_chunks(chunks) == baseline, f"第 {trial} 次随机切分失败"

print(f"\n1000 次随机切分（含空块）→ 结果全部一致 ✅  {len(DATA)} 字节 = {len(baseline)} 帧")
print("结论：切分方式与语义无关。只要按 \\n\\n 切帧，你就永远不会错位。")

1 字节/次        分成  1759 次喂入 → 13 帧 → 一致 ✅
7 字节/次        分成   252 次喂入 → 13 帧 → 一致 ✅
97 字节/次       分成    19 次喂入 → 13 帧 → 一致 ✅
4096 字节/次     分成     1 次喂入 → 13 帧 → 一致 ✅

1000 次随机切分（含空块）→ 结果全部一致 ✅  1759 字节 = 13 帧
结论：切分方式与语义无关。只要按 \n\n 切帧，你就永远不会错位。


## 4 · 陷阱二：中文会被切在字符中间

`你` 在 UTF-8 里是 **3 个字节** `e4 bd a0`。

如果 TCP 刚好切在中间，`chunk.decode("utf-8")` 会直接抛异常 ——
这是中文场景下最经典的流式线上事故，也是"明明英文能跑、中文就 500"的常见根因。

**修复方式**：不要对每个 chunk 独立 decode，要用**增量解码器**：
它会把残缺的字节尾巴含在嘴里，等下一批凑齐再吐字符。


In [12]:
s = "你好"
b = s.encode("utf-8")
print(f"{s!r} → {len(s)} 个字符，{len(b)} 个字节：{b.hex(' ')}")

head, tail = b[:2], b[2:]
print(f"\n切在中间：head = {head.hex(' ')}   tail = {tail.hex(' ')}")

# ❌ 朴素写法
try:
    head.decode("utf-8")
except UnicodeDecodeError as exc:
    print(f"\n❌ head.decode('utf-8') → UnicodeDecodeError: {exc.reason}")

# ✅ 增量写法
dec = codecs.getincrementaldecoder("utf-8")()
print("\n✅ 增量解码器：")
print(f"   decode(head) = {dec.decode(head)!r}   ← 先吞下去，不急着输出")
print(f"   decode(tail) = {dec.decode(tail)!r}  ← 凑齐了才吐出来")
print("\n一句话：解码器有状态，你不能每块单独解码。")

'你好' → 2 个字符，6 个字节：e4 bd a0 e5 a5 bd

切在中间：head = e4 bd   tail = a0 e5 a5 bd

❌ head.decode('utf-8') → UnicodeDecodeError: unexpected end of data

✅ 增量解码器：
   decode(head) = ''   ← 先吞下去，不急着输出
   decode(tail) = '你好'  ← 凑齐了才吐出来

一句话：解码器有状态，你不能每块单独解码。


## 5 · 陷阱三：`partial_json` 不能逐段解析

工具参数不是一个对象一次到齐的，它是**字符串碎片**。

原因很朴素：模型是逐 token 生成的，服务层只是把**已生成的字符原样转发**。
它在哪断开完全由 token 边界决定 —— 可能断在 `\"` 中间，可能断在数字中间，
也可能断在一个汉字的两个 UTF-8 字节之间。

下面用第 1 节那 4 段碎片验证。


In [13]:
print("① 逐段 json.loads：")
for i, frag in enumerate(FRAGMENTS):
    try:
        got = json.loads(frag)
        print(f"  [{i}] {frag!r:<34} → ✅ 居然成功了：{got!r}")
    except json.JSONDecodeError as exc:
        print(f"  [{i}] {frag!r:<34} → ❌ {exc.msg}")

print("\n② 前缀累积 json.loads：")
buf = ""
for i, frag in enumerate(FRAGMENTS):
    buf += frag
    try:
        json.loads(buf)
        print(f"  前 {i + 1} 段 ({len(buf):>2} 字符) → ✅ 可以解析了")
    except json.JSONDecodeError:
        print(f"  前 {i + 1} 段 ({len(buf):>2} 字符) → ❌ 还不完整")

print("\n③ 全部拼完：")
print("  ", json.loads("".join(FRAGMENTS)))
print("\n结论：碎片只能'攒'，不能'猜'。谁在 delta 阶段就 json.loads，谁就会随机崩。")

① 逐段 json.loads：
  [0] '{"command": "printf \'print('     → ❌ Unterminated string starting at
  [1] '\\'                               → ❌ Expecting value
  [2] '"你'                               → ❌ Unterminated string starting at
  [3] '好，world\\")\' > hello.py"}'       → ❌ Expecting value

② 前缀累积 json.loads：
  前 1 段 (27 字符) → ❌ 还不完整
  前 2 段 (28 字符) → ❌ 还不完整
  前 3 段 (30 字符) → ❌ 还不完整
  前 4 段 (54 字符) → ✅ 可以解析了

③ 全部拼完：
   {'command': 'printf \'print("你好，world")\' > hello.py'}

结论：碎片只能'攒'，不能'猜'。谁在 delta 阶段就 json.loads，谁就会随机崩。


## 6 · 拼成最终 message（这才是"真相"）

把三类增量各就各位：

| 事件 | 动作 |
|---|---|
| `content_block_start` | 按 `index` **建立**块 |
| `content_block_delta` | 往块里**追加**：文本拼字符串，工具拼 JSON 字符串 |
| `content_block_stop` | 这个块**结束**，工具参数此刻才可解析 |
| `message_delta` | 记录 `stop_reason` 与 `usage` |

> **关键认知**：`content_block_start` 里那个 `input: {}` 是**占位符**，不是答案。
> 谁在 `content_block_start` 就把 `input` 拿去执行，谁的 agent 就永远执行空命令。


In [14]:
class MessageAccumulator:
    """按 index 归并增量，还原成一条完整 message。"""

    def __init__(self) -> None:
        self.id = ""
        self.blocks: dict[int, dict] = {}
        self.stop_reason: str | None = None
        self.input_tokens = 0
        self.output_tokens = 0

    def handle(self, ev: dict) -> None:
        t = ev.get("type")

        if t == "message_start":
            self.id = ev["message"]["id"]
            u = ev["message"].get("usage") or {}
            self.input_tokens = u.get("input_tokens", 0)
            self.output_tokens = u.get("output_tokens", 0)

        elif t == "content_block_start":
            self.blocks[ev["index"]] = dict(ev["content_block"])

        elif t == "content_block_delta":
            blk = self.blocks[ev["index"]]
            d = ev["delta"]
            if d["type"] == "text_delta":
                blk["text"] = blk.get("text", "") + d["text"]
            elif d["type"] == "input_json_delta":
                blk["_json"] = blk.get("_json", "") + d["partial_json"]

        elif t == "content_block_stop":
            blk = self.blocks[ev["index"]]
            if blk.get("type") == "tool_use":
                raw = blk.pop("_json", "")
                try:
                    blk["input"] = json.loads(raw) if raw.strip() else {}
                except json.JSONDecodeError as exc:
                    blk["input"] = {}
                    blk["_parse_error"] = f"工具参数不是合法 JSON: {exc.msg}"

        elif t == "message_delta":
            self.stop_reason = (ev.get("delta") or {}).get("stop_reason")
            self.output_tokens = (ev.get("usage") or {}).get("output_tokens", self.output_tokens)

    def final(self) -> dict:
        return {
            "id": self.id,
            "role": "assistant",
            "stop_reason": self.stop_reason,
            "usage": {"input_tokens": self.input_tokens, "output_tokens": self.output_tokens},
            "content": [self.blocks[i] for i in sorted(self.blocks)],
        }


acc = MessageAccumulator()
for p in parsed:
    acc.handle(p["data"])

final = acc.final()
print(json.dumps(final, ensure_ascii=False, indent=2))

{
  "id": "msg_01DEMO",
  "role": "assistant",
  "stop_reason": "tool_use",
  "usage": {
    "input_tokens": 42,
    "output_tokens": 37
  },
  "content": [
    {
      "type": "text",
      "text": "我来创建 hello.py 并运行它。"
    },
    {
      "type": "tool_use",
      "id": "toolu_01DEMO",
      "name": "Bash",
      "input": {
        "command": "printf 'print(\"你好，world\")' > hello.py"
      }
    }
  ]
}


In [15]:
assert final["stop_reason"] == "tool_use"
assert [b["type"] for b in final["content"]] == ["text", "tool_use"]

tool = final["content"][1]
assert tool["input"] == {"command": COMMAND}, tool["input"]
assert "_parse_error" not in tool

print("文本块   :", final["content"][0]["text"])
print("工具块   :", tool["name"], tool["id"])
print("参数     :", tool["input"])
print("stop     :", final["stop_reason"], "| usage:", final["usage"])
print("\n✅ 流式事件用于展示，最终 message 用于行动。")
print("   —— 这句话的全部含义，就是上面这次 assert。")

文本块   : 我来创建 hello.py 并运行它。
工具块   : Bash toolu_01DEMO
参数     : {'command': 'printf \'print("你好，world")\' > hello.py'}
stop     : tool_use | usage: {'input_tokens': 42, 'output_tokens': 37}

✅ 流式事件用于展示，最终 message 用于行动。
   —— 这句话的全部含义，就是上面这次 assert。


## 7 · 真跑一次 HTTP：看得见的字节

上面的 `RAW` 还只是内存里的字符串。现在把它变成一个**真的 HTTP 服务**，
用三种视角各看一次：

1. **裸 socket** —— 看 HTTP 头和 chunked 编码的原样字节；
2. **`http.client`** —— 流式读取并计时，看 TTFT 和总时长；
3. **`/evil` 端点** —— 服务端故意切在汉字中间，验证你的解析器不崩。

服务端是标准库的手工 chunked，这样你能看清 `Transfer-Encoding: chunked` 到底长什么样。


In [16]:
STREAM_BYTES = RAW.encode("utf-8")

# 正常模式：一个 chunk = 一个完整帧
NORMAL_CHUNKS = [(f + "\n\n").encode("utf-8") for f in RAW.split("\n\n") if f]

# 恶意模式：切在一个汉字的中间，并把中间那 1 个字节单独发一包
_mid = STREAM_BYTES.index("你".encode("utf-8")) + 1
EVIL_CHUNKS = [STREAM_BYTES[:_mid], STREAM_BYTES[_mid:_mid + 1], STREAM_BYTES[_mid + 1:]]


class SSEHandler(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"

    def log_message(self, *args) -> None:   # 别刷屏
        pass

    def do_POST(self) -> None:
        n = int(self.headers.get("Content-Length") or 0)
        self.rfile.read(n)

        evil = "evil" in self.path
        chunks = EVIL_CHUNKS if evil else NORMAL_CHUNKS

        self.send_response(200)
        self.send_header("Content-Type", "text/event-stream; charset=utf-8")
        self.send_header("Cache-Control", "no-cache")
        self.send_header("Transfer-Encoding", "chunked")
        self.end_headers()

        try:
            time.sleep(0.05)                          # 模拟 prefill：首 token 之前要先算 prompt
            for c in chunks:
                time.sleep(0.005 if evil else 0.03)   # 模拟 decode：逐 token 生成
                self._write_chunk(c)
            self._write_chunk(b"")
        except (BrokenPipeError, ConnectionResetError):
            # 客户端提前断开 = 用户点了"停止生成"。
            # 真实引擎在这里会取消 decode，省下算力。
            pass
        self.close_connection = True

    def _write_chunk(self, data: bytes) -> None:
        self.wfile.write(f"{len(data):X}\r\n".encode("ascii"))   # 十六进制长度
        if data:
            self.wfile.write(data)
        self.wfile.write(b"\r\n")
        self.wfile.flush()


httpd = ThreadingHTTPServer(("127.0.0.1", 0), SSEHandler)
PORT = httpd.server_address[1]
threading.Thread(target=httpd.serve_forever, daemon=True).start()
print(f"mock SSE server 已启动：http://127.0.0.1:{PORT}/v1/messages")
print(f"恶意切分端点：        http://127.0.0.1:{PORT}/evil")

mock SSE server 已启动：http://127.0.0.1:46269/v1/messages
恶意切分端点：        http://127.0.0.1:46269/evil


### 7.1 裸 socket：看网线上的原样字节

这一格直接 `sendall` 一个 HTTP 请求，然后 `recv` 一次，把**收到的原始字节**打印出来。
你会看到三样东西：响应头、十六进制 chunk 长度、`data: {...}` 帧。


In [17]:
body = b'{"stream": true}'
req = (
    f"POST /v1/messages HTTP/1.1\r\n"
    f"Host: 127.0.0.1:{PORT}\r\n"
    f"Content-Type: application/json\r\n"
    f"Content-Length: {len(body)}\r\n"
    f"\r\n"
).encode("ascii") + body

with socket.create_connection(("127.0.0.1", PORT), timeout=10) as s:
    s.sendall(req)
    time.sleep(0.15)                 # 等几个 chunk 到达
    wire = s.recv(1400)

print(f"一次 recv 收到 {len(wire)} 字节（远不是完整响应）：\n")
print(wire.decode("utf-8", "replace"))
print("↑ 那些单独成行的十六进制数字（如 9f、a3）就是 chunk 长度；")
print(r"  每个 chunk 形如 '<十六进制长度>\r\n<数据>\r\n'，最后用长度 0 收尾。")

一次 recv 收到 688 字节（远不是完整响应）：

HTTP/1.1 200 OK
Server: BaseHTTP/0.6 Python/3.14.4
Date: Fri, 23 Oct 2026 13:16:08 GMT
Content-Type: text/event-stream; charset=utf-8
Cache-Control: no-cache
Transfer-Encoding: chunked

D8
event: message_start
data: {"type": "message_start", "message": {"id": "msg_01DEMO", "type": "message", "role": "assistant", "model": "demo-model", "content": [], "usage": {"input_tokens": 42, "output_tokens": 1}}}


7D
event: content_block_start
data: {"type": "content_block_start", "index": 0, "content_block": {"type": "text", "text": ""}}


88
event: content_block_delta
data: {"type": "content_block_delta", "index": 0, "delta": {"type": "text_delta", "text": "我来创建 "}}



↑ 那些单独成行的十六进制数字（如 9f、a3）就是 chunk 长度；
  每个 chunk 形如 '<十六进制长度>\r\n<数据>\r\n'，最后用长度 0 收尾。


### 7.2 用 `http.client` 流式读 + 计时

`http.client` 会自动解掉 chunked 编码，把 body 字节还给你 ——
但**它不会替你切帧，也不会替你解码 UTF-8**。这两件事仍然归你。

这一段顺便量出两个真实指标：

- **TTFT**（首帧时间）：从发请求到第一帧落地 —— 用户感受到的"多久开始出字"；
- **总时长**：全部跑完要多久。

两者的差距，就是"流式"存在的全部理由。


In [18]:
def read_stream(path: str) -> tuple[list[str], dict]:
    conn = http.client.HTTPConnection("127.0.0.1", PORT, timeout=15)
    t0 = time.perf_counter()
    conn.request("POST", path, body=json.dumps({"stream": True}),
                 headers={"Content-Type": "application/json"})
    resp = conn.getresponse()
    t_head = time.perf_counter()
    print(f"HTTP {resp.status}  content-type={resp.getheader('content-type')}"
          f"  transfer-encoding={resp.getheader('transfer-encoding')}")

    bf = ByteFramer()
    out: list[str] = []
    t_first_byte = t_first_frame = 0.0
    nbytes = 0

    while (chunk := resp.read1(4096)):
        if not t_first_byte:
            t_first_byte = time.perf_counter()
        nbytes += len(chunk)
        got = bf.feed(chunk)
        if got and not t_first_frame:
            t_first_frame = time.perf_counter()
        out += got

    out += bf.close()
    t_end = time.perf_counter()
    conn.close()
    return out, {
        "字节数": nbytes,
        "响应头": t_head - t0,
        "首帧": (t_first_frame or t_end) - t0,
        "全部结束": t_end - t0,
    }


got_frames, timing = read_stream("/v1/messages")
print(f"\n共 {len(got_frames)} 帧, {timing['字节数']} 字节")
for k in ("响应头", "首帧", "全部结束"):
    print(f"  {k:<8}{timing[k] * 1000:7.1f} ms")

assert got_frames == frames, "流式读回来的帧，必须和内存里的一致"
print("\n✅ 通过真实 HTTP 读回的帧，与本地 RAW 完全一致")

HTTP 200  content-type=text/event-stream; charset=utf-8  transfer-encoding=chunked

共 13 帧, 1759 字节
  响应头         2.9 ms
  首帧         83.4 ms
  全部结束      446.1 ms

✅ 通过真实 HTTP 读回的帧，与本地 RAW 完全一致


### 7.3 让服务端"使坏"

再请求一次 `/evil`：服务端把字节切在一个**汉字中间**，并把中间那 1 个字节单独发一包
（这个字节甚至不构成合法 UTF-8）。

如果你的解析器写对了，结果必须**一模一样**。


In [19]:
evil_frames, evil_timing = read_stream("/evil")
print(f"\n恶意切分：{len(evil_frames)} 帧, {evil_timing['字节数']} 字节")
assert evil_frames == frames, "切分方式不该影响语义"
print("结果与正常流完全一致 ✅")
print("这就是'按 \\n\\n 切帧 + 增量解码'的全部价值：")
print("  你的代码对 TCP 的切法完全免疫。")

HTTP 200  content-type=text/event-stream; charset=utf-8  transfer-encoding=chunked

恶意切分：13 帧, 1759 字节
结果与正常流完全一致 ✅
这就是'按 \n\n 切帧 + 增量解码'的全部价值：
  你的代码对 TCP 的切法完全免疫。


In [20]:
print("你也可以在终端里亲眼看一下（另开一个终端，不要关掉本 notebook）：\n")
print(f"  curl -N -X POST http://127.0.0.1:{PORT}/v1/messages \\")
print("       -H 'Content-Type: application/json' -d '{\"stream\": true}'")
print("\n关键在 -N：不加它，curl 会自己缓冲，你就看不到'流'了。")
print("同理，nginx 后面要配 proxy_buffering off，否则前端永远看不到逐字效果。")
print(f"\nmock server 仍在后台运行（daemon 线程），所以你可以随时重跑 7.x 任意一格。")
print("想回收端口：Kernel → Restart，或运行 httpd.shutdown()。")

你也可以在终端里亲眼看一下（另开一个终端，不要关掉本 notebook）：

  curl -N -X POST http://127.0.0.1:46269/v1/messages \
       -H 'Content-Type: application/json' -d '{"stream": true}'

关键在 -N：不加它，curl 会自己缓冲，你就看不到'流'了。
同理，nginx 后面要配 proxy_buffering off，否则前端永远看不到逐字效果。

mock server 仍在后台运行（daemon 线程），所以你可以随时重跑 7.x 任意一格。
想回收端口：Kernel → Restart，或运行 httpd.shutdown()。


## 8 · 翻译成 `cli_agent` 的事件

到这里你已经有了"帧 → 最终 message"。`cli_agent` 的 adapter 只多做一件事：
**在流式过程中顺便吐出归一化事件**，让 UI 能边收边显示。

对照 [`src/cli_agent/providers/anthropic_compat.py`](../src/cli_agent/providers/anthropic_compat.py) 的 `stream()`：

| 本课这一段 | 真实项目里由谁做 |
|---|---|
| `ByteFramer`（字节 → 帧） | 本地 SDK（`httpx` + 内部 SSE 解析） |
| `parse_frame`（帧 → 事件对象） | 本地 SDK（`event.type / event.delta`） |
| `MessageAccumulator` | 本地 SDK 的 `await stream.get_final_message()` |
| 下面的 `mini_adapter()` | `stream()` 里那个 `async for event in stream` 循环 |

**一处关键差别**：真实 adapter **不自己攒最终 message**，
它直接信任 `get_final_message()`。因为"累积成最终消息"是 SDK 的职责，
重复实现只会让两份逻辑慢慢漂移。

我们这一课之所以要自己攒，是因为**没有 SDK** —— 而这恰好让你看清了 SDK 到底在做什么。


In [21]:
STOP_MAP = {"end_turn": "end_turn", "tool_use": "tool_use",
            "max_tokens": "max_tokens", "refusal": "refusal",
            "content_filter": "refusal"}

if HAS_CLI_AGENT:
    from cli_agent.messages import (
        Message, TextBlock, ToolResultBlock, ToolUseBlock, Usage,
        content_block_to_dict, new_id, now_iso,
    )
    from cli_agent.providers.base import ResponseCompleted, TextDelta, ToolCall

    def mini_adapter(frames_in: list[str]):
        """SSE 帧 → ProviderEvent。形状和 anthropic_compat.stream() 一致。"""
        acc2 = MessageAccumulator()
        for frame in frames_in:
            p = parse_frame(frame)
            if p is None:
                continue
            ev = p["data"]
            acc2.handle(ev)

            if ev["type"] == "content_block_delta" and ev["delta"]["type"] == "text_delta":
                yield TextDelta(ev["delta"]["text"])
            elif ev["type"] == "content_block_stop":
                blk = acc2.blocks[ev["index"]]
                if blk.get("type") == "tool_use":
                    yield ToolCall(
                        tool_use_id=blk["id"],
                        name=blk["name"],
                        input=dict(blk["input"]),
                        parse_error=blk.get("_parse_error"),
                    )

        blocks = []
        for i in sorted(acc2.blocks):
            b = acc2.blocks[i]
            if b["type"] == "text":
                blocks.append(TextBlock(b["text"]))
            elif b["type"] == "tool_use":
                blocks.append(ToolUseBlock(b["id"], b["name"], b["input"]))
        yield ResponseCompleted(
            message=Message(id=acc2.id or new_id(), role="assistant",
                            content=tuple(blocks), created_at=now_iso()),
            stop_reason=STOP_MAP.get(acc2.stop_reason or "", "error"),
            raw_stop_reason=acc2.stop_reason,
            usage=Usage(input_tokens=acc2.input_tokens,
                        output_tokens=acc2.output_tokens),
        )

    events = list(mini_adapter(got_frames))
    print(f"共 {len(events)} 个 ProviderEvent：\n")
    for e in events:
        print(f"  {type(e).__name__:<18} {e}")
else:
    events = []
    print("跳过（cli_agent 未安装）")

共 4 个 ProviderEvent：

  TextDelta          TextDelta(text='我来创建 ')
  TextDelta          TextDelta(text='hello.py 并运行它。')
  ToolCall           ToolCall(tool_use_id='toolu_01DEMO', name='Bash', input={'command': 'printf \'print("你好，world")\' > hello.py'}, parse_error=None)
  ResponseCompleted  ResponseCompleted(message=Message(id='msg_01DEMO', role='assistant', content=(TextBlock(text='我来创建 hello.py 并运行它。'), ToolUseBlock(id='toolu_01DEMO', name='Bash', input={'command': 'printf \'print("你好，world")\' > hello.py'})), created_at='2026-10-23T13:16:08.909023+00:00'), stop_reason='tool_use', raw_stop_reason='tool_use', usage=Usage(input_tokens=42, output_tokens=37, cache_read_tokens=0, cache_write_tokens=0))


### 8.1 一个 tool call = 至少两次 HTTP 请求

`ResponseCompleted` 到了，runtime 才知道"该干活了"。它会把 `ToolUseBlock` 交给本地
`ToolExecutor` 执行，然后把结果包装成 `ToolResultBlock`，放进一条 **user message**，
作为**下一次请求**的一部分发回去。

SSE 是**单向**的：工具结果永远不走这条连接。

下面这一步就是把"第二次请求"的 wire JSON 拼出来 —— 用的正是项目自己的
`content_block_to_dict()`，所以它和真实发给 API 的形状一模一样。


In [22]:
if HAS_CLI_AGENT:
    fin = events[-1]
    assert isinstance(fin, ResponseCompleted)
    calls = fin.message.tool_uses()

    print("stop_reason :", fin.stop_reason, f"(raw={fin.raw_stop_reason})")
    print("usage       :", fin.usage)
    print("tool_uses   :", calls)

    call = calls[0]
    result_text = "你好，world\n"        # 假装 Bash 跑完了

    tool_result_msg = Message(
        id=new_id(), role="user", created_at=now_iso(),
        content=(ToolResultBlock(tool_use_id=call.id,
                                 content=(TextBlock(result_text),),
                                 is_error=False),),
    )

    wire = [
        {"role": fin.message.role,
         "content": [content_block_to_dict(b) for b in fin.message.content]},
        {"role": tool_result_msg.role,
         "content": [content_block_to_dict(b) for b in tool_result_msg.content]},
    ]
    print("\n第二次请求里的 messages（形状与真实 wire 一致）：\n")
    print(json.dumps(wire, ensure_ascii=False, indent=2))

    assert wire[0]["content"][1]["id"] == wire[1]["content"][0]["tool_use_id"]
    print("\n✅ tool_use.id 与 tool_result.tool_use_id 严格配对")
    print("   模型看不到你的 Python 对象；它看到的就是上面这段 JSON。")

stop_reason : tool_use (raw=tool_use)
usage       : Usage(input_tokens=42, output_tokens=37, cache_read_tokens=0, cache_write_tokens=0)
tool_uses   : (ToolUseBlock(id='toolu_01DEMO', name='Bash', input={'command': 'printf \'print("你好，world")\' > hello.py'}),)

第二次请求里的 messages（形状与真实 wire 一致）：

[
  {
    "role": "assistant",
    "content": [
      {
        "type": "text",
        "text": "我来创建 hello.py 并运行它。"
      },
      {
        "type": "tool_use",
        "id": "toolu_01DEMO",
        "name": "Bash",
        "input": {
          "command": "printf 'print(\"你好，world\")' > hello.py"
        }
      }
    ]
  },
  {
    "role": "user",
    "content": [
      {
        "type": "tool_result",
        "tool_use_id": "toolu_01DEMO",
        "content": [
          {
            "type": "text",
            "text": "你好，world\n"
          }
        ],
        "is_error": false
      }
    ]
  }
]

✅ tool_use.id 与 tool_result.tool_use_id 严格配对
   模型看不到你的 Python 对象；它看到的就是上面这段 JSON。


## 9 · 真调一次 DeepSeek（OpenAI 兼容协议）

前面所有实验都是**离线**的。这一节最后一次：真的发一个 HTTPS 请求，真的收一条 SSE 流。

**为什么单独讲 OpenAI 兼容？** 因为 `cli_agent` 目前只接了 Anthropic 兼容的 adapter
（`providers/anthropic_compat.py`），README 里把 "OpenAI-compatible adapter" 列在
*Deliberately deferred*。你下面要亲手做的，就是这个 adapter 的雏形。

先看两家的 wire 差异 —— 这张表在 `01_streaming_tool_call.md` §6.2 出现过，
这次我们用**真实数据**把它填满：

| 概念 | 我们前面模拟的 Anthropic 形状 | DeepSeek / OpenAI 兼容形状 |
|---|---|---|
| 文本增量 | `delta.text_delta.text` | `choices[0].delta.content` |
| 工具调用容器 | `content_block`（按 `index`） | `choices[0].delta.tool_calls[]`（按 `index`） |
| 工具参数 | `delta.input_json_delta.partial_json` | `delta.tool_calls[].function.arguments`（**JSON 字符串**） |
| 工具 id 何时出现 | `content_block_start` | 第一个 `tool_calls` 帧的 `id` |
| 结束原因 | `stop_reason: "tool_use"` | `finish_reason: "tool_calls"` |
| 结束哨兵 | `message_stop` 事件 | `data: [DONE]` |
| usage | `message_delta` 里 | 最后一帧的 `usage` 字段 |

**凭据**：优先读环境变量 `DEEPSEEK_API_KEY`；没有就退回解析 `~/.bashrc` 里的
`export DEEPSEEK_API_KEY=...`。两处都找不到时，本节所有单元自动跳过。

> **安全提示**：不要把 key 写进 notebook。这里只在运行时从环境读取，
> 不落盘、不打印完整值。要分享 notebook 前，先 `Restart` 再保存，清掉输出。


In [23]:
def _mask(s: str) -> str:
    return f"{s[:4]}…{s[-4:]}" if len(s) > 12 else "…"


def _discover_key() -> tuple[str | None, str]:
    k = os.environ.get("DEEPSEEK_API_KEY", "").strip()
    if k:
        return k, "环境变量"

    rc = pathlib.Path.home() / ".bashrc"
    if rc.exists():
        for line in rc.read_text(errors="ignore").splitlines():
            line = line.strip()
            if line.startswith("export DEEPSEEK_API_KEY="):
                value = line.split("=", 1)[1].strip().strip("\"'")
                if value:
                    return value, "~/.bashrc"
    return None, "未找到"


DEEPSEEK_KEY, KEY_SOURCE = _discover_key()
LIVE = bool(DEEPSEEK_KEY)

if LIVE:
    print(f"密钥来源：{KEY_SOURCE}")
    print(f"密钥预览：{_mask(DEEPSEEK_KEY)}   （长度 {len(DEEPSEEK_KEY)}，下面全程只用不打印）")
else:
    print("未找到 DEEPSEEK_API_KEY —— 第 9 节将自动跳过。")
    print("启用方式：export DEEPSEEK_API_KEY=sk-... 然后 Kernel → Restart")

密钥来源：环境变量
密钥预览：sk-3…1fc6   （长度 35，下面全程只用不打印）


下面这段代码**没有用 `openai` 库，也没有用 `httpx`** —— 你的环境里根本没装这两个。

只有 `http.client` + `ssl` + **第 3 节那个 `ByteFramer`**。

这恰好证明一件事：

> **分帧器不认识任何厂商，它只认识 `\n\n`。**

也就是说，第 2、3 节你写的那些代码，换一家 API 完全不用改。


In [24]:
API_HOST = "api.deepseek.com"
API_PATH = "/v1/chat/completions"


def live_request(payload: dict):
    """真发一次 HTTPS POST。标准库，没有第三方依赖。"""
    ctx = ssl.create_default_context()
    conn = http.client.HTTPSConnection(API_HOST, 443, timeout=120, context=ctx)
    conn.request(
        "POST", API_PATH,
        body=json.dumps(payload, ensure_ascii=False).encode("utf-8"),
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {DEEPSEEK_KEY}",
            "Accept": "text/event-stream",
        },
    )
    return conn, conn.getresponse()


def live_stream(payload: dict, on_frame) -> dict:
    """边收边把每一帧交给 on_frame(elapsed, frame)，返回耗时统计。"""
    t0 = time.perf_counter()          # ← 计时从"发请求之前"开始，才叫真正的 TTFT
    conn, resp = live_request(payload)
    print(f"HTTP {resp.status} {resp.reason}")
    print(f"  content-type      : {resp.getheader('content-type')}")
    print(f"  transfer-encoding : {resp.getheader('transfer-encoding')}")
    if resp.status != 200:
        print(resp.read()[:600].decode("utf-8", "replace"))
        conn.close()
        return {}

    bf = ByteFramer()                      # ← 第 3 节那个类，原封不动
    n, first = 0, None
    while (chunk := resp.read1(4096)):
        for frame in bf.feed(chunk):
            n += 1
            now = time.perf_counter() - t0
            first = now if first is None else first
            on_frame(now, frame)
    for frame in bf.close():
        n += 1
        on_frame(time.perf_counter() - t0, frame)

    total = time.perf_counter() - t0
    conn.close()
    return {"帧数": n, "首帧": first or 0.0, "总耗时": total}


class OpenAIAccumulator:
    """OpenAI 形状的累积器。关键词：delta.content 与 function.arguments。"""

    def __init__(self) -> None:
        self.id = ""
        self.model = ""
        self.text = ""
        self.tool_calls: dict[int, dict] = {}
        self.finish_reason: str | None = None
        self.usage: dict = {}

    def handle(self, c: dict) -> str:
        self.id = c.get("id", self.id)
        self.model = c.get("model", self.model)
        if c.get("usage"):
            self.usage = c["usage"]

        new_text = ""
        for ch in c.get("choices") or []:
            d = ch.get("delta") or {}
            piece = d.get("content")
            if piece:
                self.text += piece
                new_text += piece
            for tc in d.get("tool_calls") or []:
                slot = self.tool_calls.setdefault(
                    tc["index"], {"id": "", "name": "", "arguments": ""}
                )
                slot["id"] = tc.get("id") or slot["id"]
                fn = tc.get("function") or {}
                slot["name"] = fn.get("name") or slot["name"]
                slot["arguments"] += fn.get("arguments") or ""
            if ch.get("finish_reason"):
                self.finish_reason = ch["finish_reason"]
        return new_text


print("先记一个坑：")
try:
    json.loads("[DONE]")
except json.JSONDecodeError as exc:
    print(f"  json.loads('[DONE]') → JSONDecodeError: {exc.msg}")
print("  '[DONE]' 不是 JSON，它是 OpenAI 家族的结束哨兵，必须先判断再解析。")
print("  Anthropic 家族没有哨兵，它用 message_stop 事件收尾。")

先记一个坑：
  json.loads('[DONE]') → JSONDecodeError: Expecting value
  '[DONE]' 不是 JSON，它是 OpenAI 家族的结束哨兵，必须先判断再解析。
  Anthropic 家族没有哨兵，它用 message_stop 事件收尾。


### 9.1 真实文本流

一次 API 调用要花几厘钱，所以我们只问一个很短的问题。重点看四件事：

1. `content-type` 和 `transfer-encoding` 与第 7 节的 mock **一模一样**；
2. 文本是**逐块**到的，`delta.content` 每次只带一两个 token；
3. `usage` 出现在**最后一帧**（带 `finish_reason` 的那一帧），不是你想象的独立一帧；
4. 真实的 TTFT 与总耗时。


In [25]:
if LIVE:
    oa = OpenAIAccumulator()
    raws: list[str] = []
    t_first = [None]

    def on_frame(t: float, frame: str) -> None:
        if frame.strip().endswith("[DONE]"):
            return
        raws.append(frame)
        p = parse_frame(frame)
        if p is None:
            return
        if t_first[0] is None:
            t_first[0] = t
        piece = oa.handle(p["data"])
        if piece:
            print(piece, end="", flush=True)

    print("=== 流式输出开始 ===")
    stats = live_stream(
        {
            "model": "deepseek-chat",
            "messages": [
                {"role": "user", "content": "用一句话解释什么是 SSE，不超过 30 字。"}
            ],
            "stream": True,
            "max_tokens": 60,
            "stream_options": {"include_usage": True},
        },
        on_frame,
    )
    print("\n=== 流式输出结束 ===\n")

    if stats:
        print("模型     :", oa.model, "  ← 请求写的是 deepseek-chat，应答报的是这个")
        print("完整文本 :", oa.text)
        print("finish   :", oa.finish_reason)
        print("usage    :", oa.usage)
        print(f"帧数     : {stats['帧数']}")
        print(f"TTFT     : {stats['首帧'] * 1000:.0f} ms   ← 真实首字延迟（含建连 + prefill）")
        print(f"总耗时   : {stats['总耗时'] * 1000:.0f} ms")
        print("\n前 3 帧的 delta（这才是有意思的部分）：")
        for r in raws[:3]:
            d = parse_frame(r)["data"]["choices"][0]
            print("  delta =", json.dumps(d["delta"], ensure_ascii=False),
                  " finish_reason =", d.get("finish_reason"))
else:
    print("跳过：未找到 DEEPSEEK_API_KEY。")

=== 流式输出开始 ===
HTTP 200 OK
  content-type      : text/event-stream; charset=utf-8
  transfer-encoding : chunked
服务器向浏览器单向推送实时消息的技术。
=== 流式输出结束 ===

模型     : deepseek-flash   ← 请求写的是 deepseek-chat，应答报的是这个
完整文本 : 服务器向浏览器单向推送实时消息的技术。
finish   : stop
usage    : {'prompt_tokens': 16, 'completion_tokens': 9, 'total_tokens': 25, 'prompt_tokens_details': {'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 16}
帧数     : 12
TTFT     : 197 ms   ← 真实首字延迟（含建连 + prefill）
总耗时   : 1165 ms

前 3 帧的 delta（这才是有意思的部分）：
  delta = {"role": "assistant", "content": ""}  finish_reason = None
  delta = {"content": "服务器"}  finish_reason = None
  delta = {"content": "向"}  finish_reason = None


### 9.2 真实工具调用：`arguments` 也是碎片

这里是全场最值得看的一格。

Anthropic 用 `input_json_delta.partial_json` 传工具参数碎片；
OpenAI 兼容用 `tool_calls[].function.arguments` —— **一个 JSON 字符串，同样是碎片**。

而且它的切法比我们模拟的还狠：真实 token 边界会把 `tool` 切成 `t` + `ool`。
你会在输出里亲眼看到。


In [26]:
BASH_TOOL = {
    "type": "function",
    "function": {
        "name": "Bash",
        "description": "在本地机器上执行一条 shell 命令。",
        "parameters": {
            "type": "object",
            "properties": {"command": {"type": "string"}},
            "required": ["command"],
        },
    },
}

if LIVE:
    oa2 = OpenAIAccumulator()
    tool_raws: list[str] = []
    seen_args: dict[int, int] = {}

    def on_tool_frame(t: float, frame: str) -> None:
        if frame.strip().endswith("[DONE]"):
            return
        tool_raws.append(frame)
        p = parse_frame(frame)
        if p is None:
            return
        oa2.handle(p["data"])
        for i, s in oa2.tool_calls.items():
            if i not in seen_args:
                print(f"  [{i}] 声明: id={s['id']}  name={s['name']}  arguments=''"
                      f"   ← 参数此刻是空的")
                seen_args[i] = 0
            added = s["arguments"][seen_args[i]:]
            if added:
                print(f"  [{i}] 追加 {added!r:<10} → 累计 {s['arguments']!r}")
                seen_args[i] = len(s["arguments"])

    print("=== 让模型发起一次真实工具调用 ===\n")
    live_stream(
        {
            "model": "deepseek-chat",
            "messages": [{"role": "user", "content": "请在本地运行：echo hello-from-tool"}],
            "tools": [BASH_TOOL],
            "stream": True,
            "max_tokens": 300,
            "temperature": 0,
        },
        on_tool_frame,
    )

    print("\n=== 攒完之后 ===")
    for i, s in oa2.tool_calls.items():
        print(f"tool_calls[{i}]  id = {s['id']}   name = {s['name']}")
        print(f"  arguments 原始字符串 : {s['arguments']!r}")
        print(f"  json.loads 之后      : {json.loads(s['arguments'])}")
    print()
    print("finish_reason :", oa2.finish_reason,
          "  ← Anthropic 叫 'tool_use'，这里叫 'tool_calls'")
    print("注意：除了最后一帧，没有任何一帧的 arguments 能单独解析 —— 和第 5 节结论一致。")
else:
    print("跳过：未找到 DEEPSEEK_API_KEY。")

=== 让模型发起一次真实工具调用 ===

HTTP 200 OK
  content-type      : text/event-stream; charset=utf-8
  transfer-encoding : chunked
  [0] 声明: id=call_00_jxK4mPclAVDBmpD39WLk9991  name=Bash  arguments=''   ← 参数此刻是空的
  [0] 追加 '{'        → 累计 '{'
  [0] 追加 '"'        → 累计 '{"'
  [0] 追加 'command'  → 累计 '{"command'
  [0] 追加 '"'        → 累计 '{"command"'
  [0] 追加 ': '       → 累计 '{"command": '
  [0] 追加 '"'        → 累计 '{"command": "'
  [0] 追加 'echo'     → 累计 '{"command": "echo'
  [0] 追加 ' hello'   → 累计 '{"command": "echo hello'
  [0] 追加 '-from'    → 累计 '{"command": "echo hello-from'
  [0] 追加 '-t'       → 累计 '{"command": "echo hello-from-t'
  [0] 追加 'ool'      → 累计 '{"command": "echo hello-from-tool'
  [0] 追加 '"'        → 累计 '{"command": "echo hello-from-tool"'
  [0] 追加 '}'        → 累计 '{"command": "echo hello-from-tool"}'

=== 攒完之后 ===
tool_calls[0]  id = call_00_jxK4mPclAVDBmpD39WLk9991   name = Bash
  arguments 原始字符串 : '{"command": "echo hello-from-tool"}'
  json.loads 之后      : {'command': 'echo hello-

### 9.3 同一套内部事件

现在把刚才那条**真实**的 OpenAI 流，喂给一个 `openai_mini_adapter`，
让它产出和第 8 节**完全相同类型**的 `TextDelta` / `ToolCall` / `ResponseCompleted`。

对比的关键是这两个字段：

- `stop_reason` —— **内部词汇**，两家必须一致（`tool_use`）
- `raw_stop_reason` —— **厂商词汇**，原样保留（`tool_calls` vs `tool_use`）

这正是 `anthropic_compat._stop_reason()` 在做的事：
把厂商的 `refusal` / `content_filter` / `stop_sequence` … 收敛成 6 个内部值。


In [27]:
OPENAI_STOP_MAP = {
    "stop": "end_turn",
    "tool_calls": "tool_use",
    "length": "max_tokens",
    "content_filter": "refusal",
}

if LIVE and HAS_CLI_AGENT:

    def openai_mini_adapter(frames_in: list[str]):
        """真实 OpenAI 流 → ProviderEvent。和第 8 节那个 adapter 产出同一套类型。"""
        acc3 = OpenAIAccumulator()
        for frame in frames_in:
            if frame.strip().endswith("[DONE]"):
                continue
            p = parse_frame(frame)
            if p is None:
                continue
            added = acc3.handle(p["data"])
            if added:
                yield TextDelta(added)

        blocks = []
        if acc3.text:
            blocks.append(TextBlock(acc3.text))

        # OpenAI 没有"逐块结束"的信号，只能等整条流结束后再发 ToolCall。
        # 这是两家协议的一个真实差异 —— 也正是 adapter 必须吸收的东西。
        for i in sorted(acc3.tool_calls):
            s = acc3.tool_calls[i]
            parse_error = None
            try:
                parsed = json.loads(s["arguments"] or "{}")
            except json.JSONDecodeError as exc:
                parsed = {}
                parse_error = f"工具参数不是合法 JSON: {exc.msg}"
            if parse_error is None:
                blocks.append(ToolUseBlock(s["id"], s["name"], parsed))
            yield ToolCall(tool_use_id=s["id"], name=s["name"],
                           input=parsed, parse_error=parse_error)

        yield ResponseCompleted(
            message=Message(id=acc3.id or new_id(), role="assistant",
                            content=tuple(blocks), created_at=now_iso()),
            stop_reason=OPENAI_STOP_MAP.get(acc3.finish_reason or "", "error"),
            raw_stop_reason=acc3.finish_reason,
            usage=Usage(input_tokens=acc3.usage.get("prompt_tokens", 0),
                        output_tokens=acc3.usage.get("completion_tokens", 0)),
        )

    oa_events = list(openai_mini_adapter(tool_raws))
    print("真实 OpenAI 流解析出的 ProviderEvent：\n")
    for e in oa_events:
        print(f"  {type(e).__name__:<18} {e}")
    if not any(isinstance(e, TextDelta) for e in oa_events):
        print("  （这次响应没有文本块，所以没有 TextDelta —— 只有工具块）")

    oa_final = oa_events[-1]
    assert isinstance(oa_final, ResponseCompleted), oa_final
    assert oa_final.stop_reason == "tool_use", (
        f"这次模型没调工具（raw={oa_final.raw_stop_reason}），重跑本格即可"
    )
    assert oa_final.stop_reason == fin.stop_reason
    assert oa_final.raw_stop_reason != fin.raw_stop_reason

    print()
    print(" Anthropic（模拟）  内部:", fin.stop_reason, "  厂商:", repr(fin.raw_stop_reason))
    print(" OpenAI  （真实）   内部:", oa_final.stop_reason, "  厂商:",
          repr(oa_final.raw_stop_reason))
    print("\n✅ 内部词汇一致，厂商词汇不同 —— adapter 的全部工作就是吸收这个差异。")
    print("   第 8 节那段 wire JSON 换成 OpenAI 形状，runtime 一行都不用改。")
else:
    print("跳过：需要 DEEPSEEK_API_KEY + cli_agent。")

真实 OpenAI 流解析出的 ProviderEvent：

  ToolCall           ToolCall(tool_use_id='call_00_jxK4mPclAVDBmpD39WLk9991', name='Bash', input={'command': 'echo hello-from-tool'}, parse_error=None)
  ResponseCompleted  ResponseCompleted(message=Message(id='08ae9b70-61c4-45e8-a962-1710115477e3', role='assistant', content=(ToolUseBlock(id='call_00_jxK4mPclAVDBmpD39WLk9991', name='Bash', input={'command': 'echo hello-from-tool'}),), created_at='2026-10-23T13:16:10.833882+00:00'), stop_reason='tool_use', raw_stop_reason='tool_calls', usage=Usage(input_tokens=277, output_tokens=41, cache_read_tokens=0, cache_write_tokens=0))
  （这次响应没有文本块，所以没有 TextDelta —— 只有工具块）

 Anthropic（模拟）  内部: tool_use   厂商: 'tool_use'
 OpenAI  （真实）   内部: tool_use   厂商: 'tool_calls'

✅ 内部词汇一致，厂商词汇不同 —— adapter 的全部工作就是吸收这个差异。
   第 8 节那段 wire JSON 换成 OpenAI 形状，runtime 一行都不用改。


### 9.4 这一节真正证明了什么

| 我们验证过的事 | 对应到 `01_streaming_tool_call.md` 的哪句话 |
|---|---|
| 换一家厂商，`ByteFramer` / `SSEFramer` 一行没改 | "SDK 解析的是 SSE 帧，不是 token 流" |
| `arguments` 逐字符到达，必须攒到最后 | "不要自己逐段 `json.loads`" |
| 内部 `stop_reason` 一致、`raw` 保留 | "`_stop_reason` 把未知值收敛成 `error` + `raw_stop_reason`" |
| 工具调用只是"声明"，执行还在地面 | "模型没有手。它只是写了一张工单" |
| 一次调用一个 `usage` | "usage 是服务端留给你的计费小票" |

**所以 adapter 存在的唯一理由，是让 wire 差异止步于 `providers/` 一个目录。**
你刚刚手写的这 30 行，就是 `providers/openai_compat.py` 的骨架。

> 输出说明：这一节的日志是运行时快照。重跑会变（模型输出、计时、trace id 都会变）。


## 10 · 自测（先自己答，再展开）

<details>
<summary>1. 一个 SSE 帧的边界是什么？为什么不能按"一次 read"切？</summary>

边界是空行（`\n\n` 或 `\r\n\r\n`）。TCP 只保证字节顺序，不保证你一次 `read()`
拿到的正好是一个帧 —— 可能半个，也可能三个半。按 read 切必然错位。

</details>

<details>
<summary>2. 为什么不能对每个 chunk 单独 `decode("utf-8")`？</summary>

一个汉字在 UTF-8 里占 3 字节，chunk 边界可能落在中间，此时那段字节不是合法 UTF-8。
要用增量解码器（`codecs.getincrementaldecoder`），它会暂存残缺尾巴。

</details>

<details>
<summary>3. `content_block_start` 里的 <code>"input": {}</code> 是什么？</summary>

**占位符**。真正的参数由后续 `input_json_delta` 的 `partial_json` 碎片拼成，
到 `content_block_stop` 才完整。在 start 阶段就执行 = 永远执行空参数。

</details>

<details>
<summary>4. 为什么 `input_json_delta` 不能逐段 `json.loads`？</summary>

因为碎片可能在任意位置断开：断在 `\"` 中间就产生一个孤立的 `\`；断在数字中间就产生
半个数字。任何"尽力而为"的解析都只能靠猜。正确做法是累积到 `content_block_stop`。

</details>

<details>
<summary>5. 流式 delta 和最终 message，哪个用于执行？为什么？</summary>

最终 message。流可能中断、JSON 可能被截断；只有完整到达的 message 是经过累积校验的。
**流式事件用于展示，最终 message 用于行动。**

</details>

<details>
<summary>6. 工具结果从哪里回给模型？</summary>

**不是**同一条 SSE 连接。SSE 是单向的。工具结果被包装成 `ToolResultBlock`，
放进一条 `role="user"` 的 message，作为**下一次全新 HTTP POST** 的一部分发出去。

</details>

<details>
<summary>7. 一个 tool call 至少几次 HTTP 请求？</summary>

两次。第一次模型说"我要调这个工具"，第二次把工具结果送回去让模型继续。
如果模型连续调多个工具，就是 N+1 次。

</details>

<details>
<summary>8. 为什么 curl 要加 <code>-N</code>？</summary>

不加它，curl 会缓冲响应体直到结束，你就看不到"流"。同理，反向代理要关掉 buffering。
这是"我代码写对了但看不到逐字效果"的头号原因。

</details>

<details>
<summary>9. <code>data: [DONE]</code> 为什么会让 <code>parse_frame</code> 崩？</summary>

因为它**不是 JSON**，而 `parse_frame` 无条件做了 `json.loads`。它是 OpenAI 家族的
结束哨兵，必须在解析前单独判断。Anthropic 家族没有哨兵，靠 `message_stop` 事件收尾。

**这就是"协议没统一"的具体代价，也是 adapter 必须存在的原因。**

</details>

<details>
<summary>10. 换厂商时，哪一层要改、哪一层不用改？</summary>

**要改**：`providers/openai_compat.py`（新增一个文件）—— 字段名、`[DONE]` 哨兵、
`finish_reason` 取值、`arguments` 是字符串而非对象。

**不用改**：`runtime.py`、`context/`、`session/`、`tools/`、`permissions/`，
以及整个 UI。因为它们只认内部 `Message` / `ProviderEvent`。

</details>


## 11 · 下一站

现在去读 [`01_streaming_tool_call.md`](01_streaming_tool_call.md)，请带着这三个问题：

1. `AnthropicCompatProvider.stream()` 里，**哪些行**对应本课的 `mini_adapter()`，**哪些行**
   是本课没写、因为交给了 SDK 的？
2. 为什么它处理的是 `content_block_delta` / `content_block_stop`，而不是 SDK 提供的
   `TextEvent` / `InputJsonEvent` 这类便利事件？
3. 如果网络在 `input_json_delta` 中途断开，`ResponseCompleted` 就不会到达 ——
   runtime 会执行工具吗？为什么这个设计是安全的？

### 复跑

本 notebook 完全离线、可重复运行。想重置：`Kernel → Restart & Run All`。

### 想再往前走一步

把第 7 节的 mock server 改成**两轮**：第一轮返回工具调用，收到 `tool_result` 后
第二轮返回 `stop_reason="end_turn"`。再用 `mini_adapter` 跑一遍，你就手写了一个
最小的 agent 循环。
